In [ ]:
import pandas as pd
import numpy as np
import random
import math

# Recipe class


In [ ]:
class Recipe:
    def __init__(self, RID, recipes_df):
        """
        Get recipe information from recipes_df and set attributes:
            self.RID
            self.name
            self.type
            self.category
            self.total_price
            self.provided_calories
            self.provided_protein
            self.provided_fat
            self.provided_carbs
        
        """
        recipe_row = recipes_df[recipes_df['RID'] == RID].iloc[0]
        
        self.RID = RID
        self.name = recipe_row['Name']
        self.type = recipe_row['type']
        self.category = recipe_row['Category']
        self.total_price = recipe_row['total_price']
        self.provided_calories = recipe_row['provided_calories']
        self.provided_protein = recipe_row.get('provided_protein', 0)
        self.provided_fat = recipe_row.get('provided_fat', 0)
        self.provided_carbs = recipe_row.get('provided_carbs', 0)

    def __eq__(self, other):
        """Check equality based on RID"""
        return isinstance(other, Recipe) and self.RID == other.RID


### initialising recipes list

In [ ]:
def initialize_recipes(recipes_df):
    """
    Parameter:
        recipes_df: pandas DataFrame of all recipes in the dataset

    Return:
        a list of Recipe objects
    """
    recipes_list = []
    for rid in recipes_df['RID']:
        recipes_list.append(Recipe(rid, recipes_df))
    return recipes_list


recipes_df = pd.read_csv("./data/recipes.csv")
recipes = initialize_recipes(recipes_df)

# Node class

In [ ]:
class MealPlannerState:
    def __init__(self, day_number, meal_type, recipe_id, remaining_budget, today_calorie_use, used_meals):
        """       
        Parameters:
            day_number: the current day
            meal_type: the meal we're searching
            recipe_id: the chosen recipe
            remaining_budget: remaining total budget
            today_calorie_use: the usage in the current day 
            used_meals: set of used meals this week and their repititions

        sets attributes:
            self.day
            self.meal_type
            self.recipe_id
            self.remaining_budget
            self.today_calorie_use
            self.used_meals
        """
        self.day = day_number
        self.meal_type = meal_type
        self.recipe_id = recipe_id
        self.remaining_budget = remaining_budget
        self.today_calorie_use = today_calorie_use
        self.used_meals = used_meals

    def __eq__(self, other):
        return isinstance(other, MealPlannerState) and \
            self.day == other.day and \
            self.meal_type == other.meal_type and \
            self.recipe_id == other.recipe_id

    def __hash__(self):
        return hash((self.day, self.meal_type, self.recipe_id))
        

In [ ]:
class Node:    
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        """       
        Parameters:
            state: a MealPlannerState object. represents a slot
            parent: Parent Node that generated this node (None for root)
            action: recipe_id
            cost: cost of the action leading here.
            heuristic: heuristic value of this node 

        sets attributes:
            self.state
            self.parent
            self.action
            self.g: Cummulative cost of the path to this node.
            self.f: Evaluation function
            self.depth: 0 for root, or parent.depth + 1 for children.
        """
        self.state = state
        self.parent = parent
        self.action = action
        self.g = (parent.g + cost) if parent else 0
        self.f = self.g + heuristic
        self.depth = 0 if parent is None else parent.depth + 1

    def __gt__(self, other):
        """
        Compare nodes for priority queue ordering.
        Return True if this node's f value is greater than other's f value.
        Used by search algorithms with PriorityQueue.
        """
        return self.f > other.f

    def __lt__(self, other):
        """
        Compare nodes for priority queue ordering.
        Return True if this node's f value is less than other's f value.
        Used by search algorithms with PriorityQueue.
        """
        return self.f < other.f

    def __eq__(self, other):
        """
        Check equality based on state.
        Two nodes are equal if they represent the same meal plan state.
        """
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        """
        Compute hash based on the state.
        This allows nodes to be used in sets and as dict keys.
        """
        return hash(self.state)


# Problem class

In [ ]:
class MealPlanningProblem:

    def __init__(self, recipes, daily_calories_need, total_budget, num_days=7):
        """
        Initialize the meal planning problem.
        
        Parameters:
            recipes: list of Recipe objects
            daily_calories_need
            total_budget
            num_days: Number of days to plan for
        
        sets attributes:
            self.recipes: dict mapping recipe RID to Recipe objects
            self.total_budget
            self.daily_calories_need
            self.num_days
            self.total_slots
            self.initial_state
        """
        self.recipes = {recipe.RID: recipe for recipe in recipes}
        self.total_budget = total_budget
        self.daily_calories_need = daily_calories_need
        self.num_days = num_days
        self.meals_per_day = 3  # Breakfast, Lunch, Dinner
        self.total_slots = num_days * self.meals_per_day
        self.meal_types = ['Breakfast', 'Lunch', 'Dinner']
        # Meal type weights: percentage of daily budget/calories allocated to each meal
        self.meal_type_weights = {
            'Breakfast': 0.25,
            'Lunch': 0.40,
            'Dinner': 0.35
        }
        # Initial state:  day_number, meal_type, recipe_id, remaining_budget, today_calorie_use, used_meals
        self.initial_state = MealPlannerState(0, 0, None, total_budget, 0, set())
        
        # Cache average values for heuristic calculations
        recipes_values = list(self.recipes.values())
        self._avg_price = sum(r.total_price for r in recipes_values) / len(recipes_values)
        self._avg_calories = sum(r.provided_calories for r in recipes_values) / len(recipes_values)

    def _get_allocated_budget(self, remaining_days, remaining_budget, meal_type):
        """Get the allocated budget for a specific meal type based on remaining budget"""
        weight = self.meal_type_weights[meal_type]
        return (remaining_budget / max(1, remaining_days)) * weight

    def _get_allocated_calories(self, meal_type):
        """Get the allocated calories for a specific meal type"""
        weight = self.meal_type_weights[meal_type]
        return self.daily_calories_need * weight

    def _get_remaining_days(self, state):
        current_slot = state.day * self.meals_per_day + state.meal_type + 1
        remaining_slots = self.total_slots - current_slot
        return math.ceil(remaining_slots / self.meals_per_day)

    def filter_recipes(self, meal_type=None, category=None, max_price=None, max_calories=None):
        """
        Filter recipes based on multiple criteria.
        
        Parameters:
            meal_type: Filter by meal type (str, case-insensitive)
            category: Filter by category (str, case-insensitive)
            max_price: Filter recipes with price <= max_price
            max_calories: Filter recipes with calories <= max_calories
        
        Returns: List of recipe IDs matching all criteria
        """
        filtered = []
        for rid, recipe in self.recipes.items():
            if meal_type and recipe.type.lower() != meal_type.lower():
                continue
            if category and recipe.category.lower() != category.lower():
                continue
            if max_price is not None and recipe.total_price > max_price:
                continue
            if max_calories is not None and recipe.provided_calories > max_calories:
                continue
            filtered.append(rid)
        return filtered

    def _get_all_recipes_from_path(self, node):
        """Helper: traverse parent chain to get all recipes chosen so far"""
        recipes = []
        current = node
        while current.parent is not None:
            recipes.append(current.action)
            current = current.parent
        recipes.reverse()
        return recipes

    def _calculate_totals(self, recipes_list):
        """Helper: calculate total cost, calories, and macros from recipe list"""
        total_cost = 0
        total_calories = 0
        total_protein = 0
        total_fat = 0
        total_carbs = 0
        
        for recipe_id in recipes_list:
            recipe = self.recipes[recipe_id]
            total_cost += recipe.total_price
            total_calories += recipe.provided_calories
            total_protein += recipe.provided_protein
            total_fat += recipe.provided_fat
            total_carbs += recipe.provided_carbs
        
        return total_cost, total_calories, total_protein, total_fat, total_carbs

    def is_goal(self, node):
        """
        Check if a meal plan is complete and meets nutritional goals.
        
        Should verify:
        1- All day/meal slots are filled (no empty slots)
        2- Calories are within 10% of goal
        3- Didn't go over the budget
        
        Parameters:
            node: Node representing the current state
        
        Returns: True if node represents a valid goal state, False otherwise
        """
        state = node.state
        
        # Goal reached when we've moved past the last slot
        if state.day < self.num_days:
            return False
        
        # Get all recipes chosen on the path to this node
        recipes_chosen = self._get_all_recipes_from_path(node)
        
        # All slots should be filled
        if len(recipes_chosen) != self.total_slots:
            return False
        
        # Validate constraints
        total_cost, total_calories, total_protein, total_fat, total_carbs = self._calculate_totals(recipes_chosen)
        
        # Check calories within 10% of goal
        goal_calories = self.daily_calories_need * self.num_days
        calorie_tolerance = goal_calories * 0.1
        calories_ok = abs(total_calories - goal_calories) <= calorie_tolerance
        
        # Check budget constraint (should be implicit, but verify)
        budget_ok = total_cost <= self.total_budget
        
        return calories_ok and budget_ok

    def get_valid_actions(self, state):
        """
        Return list of recipe IDs that can be added to the next meal slot.
        Filters recipes by the meal type of the current slot.
        
        Returns: List of recipe IDs matching the meal type
        """
        meal_type = self.meal_types[state.meal_type]
        return self.filter_recipes(meal_type=meal_type)

    def expand_node(self, node, use_cost=True, use_heuristic=False):
        """
        Generate child nodes by adding each valid recipe to the current node.

        Parameters:
            node: Node to expand
            use_cost: Whether to include cost in f value
            use_heuristic: Whether to include heuristic in f value
        
        Returns: List of child Node objects (empty list if we're done)
        """
        state = node.state
        
        if state.day >= self.num_days:
            return []
        
        children = []
        valid_actions = self.get_valid_actions(node.state)
        
        for recipe_id in valid_actions:

            if recipe_id in state.used_meals:
                continue

            recipe = self.recipes[recipe_id]
            
            # Calculate cost of adding this recipe
            action_cost = self.calculate_cost(node.state, recipe_id) if use_cost else 0
            
            # Calculate next state
            new_meal_idx = (state.meal_type + 1) % self.meals_per_day
            new_day = state.day if new_meal_idx > 0 else state.day + 1
            new_remaining_budget = state.remaining_budget - recipe.total_price
            new_calorie_use = 0 if new_meal_idx == 0 else state.today_calorie_use + recipe.provided_calories
            new_used_meals = set() if new_day % 4 == 0 else set(state.used_meals)
            new_used_meals.add(recipe_id)
            
            new_state = MealPlannerState(new_day, new_meal_idx, recipe_id, new_remaining_budget, new_calorie_use, new_used_meals)
            
            # Calculate heuristic estimate
            heuristic = self.calculate_heuristic(new_state) if use_heuristic else 0
            
            # Create child node
            child = Node(new_state, parent=node, action=recipe_id, cost=action_cost, heuristic=heuristic)
            children.append(child)
        
        return children

    def calculate_cost(self, state, recipe_id=None):
        """
        Calculate cost of adding a recipe to the current state.
        
        Cost combines normalized price and calorie deviations from allocated values.
        
        Parameters:
            state: Current state (day, meal_idx, recipe_id, remaining_budget)
            recipe_id: If provided, use this recipe; else use recipe_id from state
        
        Returns: Combined deviation cost (always >= 0)
        """
        
        if recipe_id is None:
            recipe_id = state.recipe_id
        
        if recipe_id is None:
            return 0
        
        recipe = self.recipes[recipe_id]
        meal_type = self.meal_types[state.meal_type]
        
        # Calculate remaining days for budget allocation
        remaining_days = self._get_remaining_days(state)
        
        # Get allocated values for this meal type using remaining budget
        allocated_price = self._get_allocated_budget(remaining_days, state.remaining_budget, meal_type)
        allocated_calories = self._get_allocated_calories(meal_type)
        
        # Calculate absolute deviations with normalization to same scale
        price_deviation = abs(recipe.total_price - allocated_price)
        calorie_deviation = abs(recipe.provided_calories - allocated_calories)
        
        # Normalize both to comparable ranges (always non-negative)
        normalized_price_dev = price_deviation / allocated_price if allocated_price > 0 else 0
        normalized_cal_dev = calorie_deviation / allocated_calories if allocated_calories > 0 else 0
        
        # Combined cost (equal weight, always non-negative)
        total_cost = (normalized_price_dev + normalized_cal_dev) / 2.0
        
        return total_cost

    def calculate_heuristic(self, state):
        """
        Estimate cost of remaining slots to reach goal.
        
        Uses average recipe values and allocated budgets to estimate
        how much deviation cost will be incurred by remaining slots.
        
        Returns: Estimated remaining cost (lower = closer to goal)
        """
        
        # Remaining slots to fill
        remaining_slots = self.total_slots - (state.day * self.meals_per_day + state.meal_type)
        
        if remaining_slots <= 0:
            return 0
        
        # Estimate total deviation for remaining meals
        total_estimated_cost = 0
        
        # Distribute remaining slots across meal types
        remaining_days = self._get_remaining_days(state)
        current_day = state.day
        current_meal = state.meal_type
        current_remaining_budget = state.remaining_budget
        
        for i in range(remaining_slots):
            meal_type = self.meal_types[current_meal]
            
            # Prevent budget from going negative
            current_remaining_budget = max(0, current_remaining_budget)
            
            # Get allocated values for this meal using current remaining budget
            allocated_price = self._get_allocated_budget(remaining_days, current_remaining_budget, meal_type)
            allocated_calories = self._get_allocated_calories(meal_type)
            
            # Estimate deviation using average recipe values with normalization
            avg_price_dev = abs(self._avg_price - allocated_price)
            avg_cal_dev = abs(self._avg_calories - allocated_calories)
            
            # Normalize both to comparable ranges (always non-negative)
            normalized_price_dev = avg_price_dev / allocated_price if allocated_price > 0 else 0
            normalized_cal_dev = avg_cal_dev / allocated_calories if allocated_calories > 0 else 0
            
            # Add to total (always non-negative)
            total_estimated_cost += (normalized_price_dev + normalized_cal_dev) / 2.0
            
            # Update remaining budget estimate safely
            current_remaining_budget = max(0, current_remaining_budget - self._avg_price)
            
            # Move to next meal slot
            current_meal = (current_meal + 1) % self.meals_per_day
            if current_meal == 0:
                current_day += 1
                remaining_days = self._get_remaining_days(state)
        
        return total_estimated_cost


# Search class

In [ ]:
class MealPlannerSearch:   # search classes inherit this one
    def __init__(self, problem):
        """       
        Parameters:
            problem: MealPlanningProblem instance
        
        Should initialize:
            self.problem
        """
        self.problem = problem

    def search(self):
        """
            does the search and returns the path (list of actions a.k.a recipe_ids)
            overwritten by child classes AstarSearch and GreedySearch        
        """
        pass

    def get_solution_path(self, solution_node):
        """
        Parameters:
            solution_node: Node representing goal state
        
        Returns: List of actions [recipe_id1, ...] representing the sequence of recipes added to reach goal
        """
        path = []
        current = solution_node
        
        while current.parent is not None:
            path.append(current.action)
            current = current.parent
        
        path.reverse()
        return path

    def print_solution(self, recipe_sequence):
        """
        print the solution meal plan including:
            Organized plan by day and meal type with recipe names
            Total cost
            Total nutrition value
        """
                
        print("\n" + "="*60)
        print("MEAL PLAN SOLUTION")
        print("="*60)
        
        # Organize by day and meal type
        meal_types = ['Breakfast', 'Lunch', 'Dinner']
        total_cost = 0
        largest_calories_deviation = 0
        total_protein = 0
        total_fat = 0
        total_carbs = 0
        
        recipe_idx = 0
        for day in range(self.problem.num_days):
            print(f"\nDay {day + 1}:")
            day_calorie_intake = 0
            for meal_type_idx, meal_type in enumerate(meal_types):
                if recipe_idx < len(recipe_sequence):
                    recipe_id = recipe_sequence[recipe_idx]
                    recipe = self.problem.recipes[recipe_id]
                    print(f"  {meal_type}: {recipe.name} (${recipe.total_price:.2f}, {recipe.provided_calories} cal)")
                    
                    total_cost += recipe.total_price

                    day_calorie_intake += recipe.provided_calories

                    total_protein += recipe.provided_protein
                    total_fat += recipe.provided_fat
                    total_carbs += recipe.provided_carbs
                    
                    recipe_idx += 1

            deviation = self.problem.daily_calories_need - day_calorie_intake
            percentage = deviation * 100 / self.problem.daily_calories_need

            largest_calories_deviation = percentage if percentage > largest_calories_deviation else largest_calories_deviation

            print(f"\nCalories intake: {day_calorie_intake}\ndeviation: {deviation}\npercentage: {percentage}")
        
        print("\n" + "="*60)
        print("SUMMARY")
        print("="*60)
        print(f"Total Cost: ${total_cost:.2f}")
        print(f"Largest Calories Deviation: {largest_calories_deviation} %")
        print(f"Total Protein: {total_protein:.2f}g")
        print(f"Total Fat: {total_fat:.2f}g")
        print(f"Total Carbs: {total_carbs:.2f}g")
        print(f"Budget Limit: ${self.problem.total_budget:.2f}")
        print(f"Daily Calorie Goal: {self.problem.daily_calories_need} cal")
        print("="*60 + "\n")


In [ ]:
import queue

class AstarSearch(MealPlannerSearch):
    def __init__(self,problem):
        super().__init__(problem)
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self.get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, True, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)


        

In [ ]:
def test_a_star(TDEE, budget, days):
    problem = MealPlanningProblem(recipes, TDEE, budget, days)
    A_star = AstarSearch(problem)

    solution = A_star.search()

    if solution is None:
        print("couldn't find a suitable plan")
        return

    A_star.print_solution(solution)
    

In [ ]:
##### TESTING #####
test_a_star(2200, 4500, 7)


MEAL PLAN SOLUTION

Day 1:
  Breakfast: Chickpea Tomato Breakfast Stew ($69.60, 447.68 cal)
  Lunch: Berkoukes ($235.02, 1142.64 cal)
  Dinner: Grilled Salmon Plate ($229.30, 337.34 cal)

Calories intake: 1927.66
deviation: 272.3399999999999
percentage: 12.379090909090905

Day 2:
  Breakfast: Daurade Olive Breakfast Bowl ($241.66, 224.45 cal)
  Lunch: Tuna Lettuce Mint Bowl ($252.90, 239.6 cal)
  Dinner: Rechta ($233.90, 1436.24 cal)

Calories intake: 1900.29
deviation: 299.71000000000004
percentage: 13.62318181818182

Day 3:
  Breakfast: Tuna Olive Breakfast Salad ($230.35, 179.05 cal)
  Lunch: Tuna Salad Bowl ($213.99, 296.81 cal)
  Dinner: Dolma ($248.58, 788.24 cal)

Calories intake: 1264.1
deviation: 935.9000000000001
percentage: 42.540909090909096

Day 4:
  Breakfast: Coffee ($10.00, 0.1 cal)
  Lunch: Rice with Peas and Carrots ($65.70, 858.1 cal)
  Dinner: Green Bean Rice Bowl ($68.06, 786.7 cal)

Calories intake: 1644.9
deviation: 555.0999999999999
percentage: 25.2318181818181